# Sistema de Recomendación de Productos — Entrega 2
**Proyecto DSA 2026** | Jessica Salazar · Juan Camilo Umaña

---

## Pregunta de negocio
> *¿Qué productos se le deberían recomendar a cada cliente según su historial de compra y sus características comerciales, con el fin de aumentar la probabilidad de compra y la activación del portafolio?*

### Enfoque metodológico
- **Variable objetivo:** binaria — ¿el cliente compró el producto en el siguiente mes? (1 = sí, 0 = no)
- **Unidad de análisis:** combinación (cliente, producto, mes)
- **Salida:** ranking Top-3 de productos recomendados por cliente
- **Validación temporal:** último mes disponible como conjunto de prueba
- **Versioning:** experimentos registrados con MLflow

## 0. Librerías

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    average_precision_score, roc_curve, precision_recall_curve
)
import mlflow
import mlflow.sklearn

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
print('Librerías cargadas correctamente.')

## 1. Carga de datos

Se trabaja con dos fuentes:
- **Clientes.csv** — características comerciales de 6,316 clientes
- **Resumen.csv** — 84,676 transacciones de ventas (enero–mayo 2026, 12 productos)

In [ ]:
clientes = pd.read_csv('Clientes.csv', encoding='latin1', sep=';')
resumen  = pd.read_csv('Resumen.csv',  encoding='latin1', sep=None, engine='python')

# Formato numérico colombiano: punto=miles, coma=decimal
for col in ['Venta', 'KG', 'Ton']:
    resumen[col] = (
        resumen[col].astype(str)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .astype(float)
    )

clientes.rename(columns={'Deudor': 'Cliente'}, inplace=True)

print(f'Clientes : {clientes.shape[0]:,} filas | {clientes.shape[1]} columnas')
print(f'Resumen  : {resumen.shape[0]:,} filas | {resumen.shape[1]} columnas')
print(f'Meses    : {sorted(resumen["Mes"].unique())}')
print(f'Productos: {sorted(resumen["CodProducto"].unique())}')
display(resumen.head(3))
display(clientes.head(3))

## 2. Ingeniería de características

Se construye un grid completo de todas las combinaciones (mes, cliente, producto) y se calculan features basadas en el historial de compra:

| Feature | Descripción |
|---|---|
| `compro_mes_ant` | ¿Compró ese producto el mes anterior? (0/1) |
| `veces_comprado` | Meses históricos con compra del producto |
| `freq_compra` | Proporción de meses con compra sobre meses disponibles |
| `ventas_cliente_ant` | Ventas totales del cliente en el mes anterior |
| `productos_ant` | Productos distintos comprados el mes anterior |
| `Frecuencia de visita` | Frecuencia de visita comercial |
| `oficina_enc` | Oficina de ventas (codificada) |
| `seg5_enc` / `seg4_enc` | Segmentos comerciales (codificados) |
| `producto_enc` | Producto (codificado) |

In [ ]:
meses_ordenados = sorted(resumen['Mes'].unique())
productos       = sorted(resumen['CodProducto'].unique())
todos_clientes  = resumen['Cliente'].unique()

resumen['compro'] = (resumen['Venta'] > 0).astype(int)

pivot = (
    resumen.groupby(['Mes', 'Cliente', 'CodProducto'])['compro']
    .max().reset_index()
)

# Grid completo
grid = pd.MultiIndex.from_product(
    [meses_ordenados, todos_clientes, productos],
    names=['Mes', 'Cliente', 'CodProducto']
)
grid_df = pd.DataFrame(index=grid).reset_index()
grid_df = grid_df.merge(pivot, on=['Mes', 'Cliente', 'CodProducto'], how='left')
grid_df['compro'] = grid_df['compro'].fillna(0).astype(int)
grid_df = grid_df.sort_values(['Cliente', 'CodProducto', 'Mes']).reset_index(drop=True)

# Features temporales
grid_df['compro_mes_ant'] = (
    grid_df.groupby(['Cliente', 'CodProducto'])['compro'].shift(1).fillna(0).astype(int)
)
grid_df['veces_comprado'] = (
    grid_df.groupby(['Cliente', 'CodProducto'])['compro']
    .transform(lambda x: x.shift(1).fillna(0).cumsum())
)
mes_idx = {m: i+1 for i, m in enumerate(meses_ordenados)}
grid_df['mes_idx']     = grid_df['Mes'].map(mes_idx)
grid_df['freq_compra'] = grid_df['veces_comprado'] / grid_df['mes_idx'].clip(lower=1)

ventas_mes = (
    resumen.groupby(['Mes', 'Cliente'])['Venta'].sum().reset_index()
    .rename(columns={'Venta': 'ventas_cliente_ant', 'Mes': 'Mes_ant'})
)
grid_df['Mes_ant'] = grid_df.groupby(['Cliente', 'CodProducto'])['Mes'].shift(1)
grid_df = grid_df.merge(ventas_mes, on=['Mes_ant', 'Cliente'], how='left')
grid_df['ventas_cliente_ant'] = grid_df['ventas_cliente_ant'].fillna(0)

prods_mes = (
    resumen[resumen['Venta'] > 0]
    .groupby(['Mes', 'Cliente'])['CodProducto'].nunique().reset_index()
    .rename(columns={'CodProducto': 'productos_ant', 'Mes': 'Mes_ant'})
)
grid_df = grid_df.merge(prods_mes, on=['Mes_ant', 'Cliente'], how='left')
grid_df['productos_ant'] = grid_df['productos_ant'].fillna(0)
grid_df['target'] = grid_df['compro']

# Codificación de clientes
le_oficina  = LabelEncoder()
le_seg5     = LabelEncoder()
le_seg4     = LabelEncoder()
le_producto = LabelEncoder()

clientes['oficina_enc'] = le_oficina.fit_transform(clientes['Oficina de ventas'])
clientes['seg5_enc']    = le_seg5.fit_transform(clientes['Denominación Gr#Clientes 5'])
clientes['seg4_enc']    = le_seg4.fit_transform(clientes['Denominación Gr#Clientes 4'])

grid_df = grid_df.merge(
    clientes[['Cliente', 'Frecuencia de visita', 'oficina_enc', 'seg5_enc', 'seg4_enc']],
    on='Cliente', how='left'
)
grid_df['producto_enc'] = le_producto.fit_transform(grid_df['CodProducto'])
grid_df.fillna(0, inplace=True)

print(f'Dataset: {grid_df.shape[0]:,} filas | Tasa positivos: {grid_df["target"].mean():.2%}')
display(grid_df[['Mes','Cliente','CodProducto','freq_compra','veces_comprado','target']].head(8))

## 3. División Train / Test — validación temporal

El **último mes (202605)** se reserva como test. Los meses anteriores (202602–202604) se usan para entrenamiento. Este enfoque replica el uso real del modelo: entrenamos con lo pasado y predecimos el futuro.

In [ ]:
FEATURES = [
    'compro_mes_ant', 'veces_comprado', 'freq_compra',
    'ventas_cliente_ant', 'productos_ant',
    'Frecuencia de visita', 'oficina_enc', 'seg5_enc', 'seg4_enc', 'producto_enc',
]

ultimo_mes = meses_ordenados[-1]
df_model   = grid_df[grid_df['Mes'] >= meses_ordenados[1]].copy()
mask_test  = df_model['Mes'] == ultimo_mes

X_train = df_model.loc[~mask_test, FEATURES]
X_test  = df_model.loc[ mask_test, FEATURES]
y_train = df_model.loc[~mask_test, 'target']
y_test  = df_model.loc[ mask_test, 'target']

print(f'Train : {X_train.shape[0]:,} filas | Positivos: {y_train.mean():.2%}')
print(f'Test  : {X_test.shape[0]:,} filas  | Positivos: {y_test.mean():.2%}')
print(f'Mes de prueba: {ultimo_mes}')

## 4. Experimentos con MLflow

Se entrenan tres modelos de complejidad creciente. Todos los experimentos quedan registrados en MLflow (parámetros, métricas y modelos serializados).

| Modelo | Tipo |
|---|---|
| Regresión Logística | Lineal — baseline |
| Random Forest | Ensamble de árboles |
| Gradient Boosting | Boosting secuencial |

In [ ]:
# ── Conexión a MLflow en EC2 ────────────────────────────────────────────────
# IMPORTANTE: reemplaza la IP por la IPv4 pública de tu instancia EC2.
# La encuentras en: AWS Console → EC2 → Instancias → columna 'IPv4 pública'
# El servidor MLflow debe estar corriendo en EC2 con:
#   mlflow server --host 0.0.0.0 --port 5000 \
#                 --backend-store-uri sqlite:///mlflow.db \
#                 --default-artifact-root ./mlartifacts
# ────────────────────────────────────────────────────────────────────────────
EC2_IP = '54.123.45.67'   # <── CAMBIA ESTO por la IP pública de tu EC2
mlflow.set_tracking_uri(f'http://{EC2_IP}:5000')
mlflow.set_experiment('Recomendacion_Productos_DSA2026')

modelos_cfg = {
    'LogisticRegression': LogisticRegression(
        max_iter=500, class_weight='balanced', random_state=42),
    'RandomForest': RandomForestClassifier(
        n_estimators=200, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
}

resultados = {}
for nombre, modelo in modelos_cfg.items():
    print(f'Entrenando {nombre}...')
    with mlflow.start_run(run_name=nombre):
        mlflow.log_params(modelo.get_params())
        mlflow.log_param('features', str(FEATURES))
        mlflow.log_param('train_size', len(X_train))
        mlflow.log_param('test_size',  len(X_test))
        mlflow.log_param('mes_test',   str(ultimo_mes))
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)
        y_prob = modelo.predict_proba(X_test)[:, 1]
        metricas = {
            'roc_auc'  : roc_auc_score(y_test, y_prob),
            'pr_auc'   : average_precision_score(y_test, y_prob),
            'f1'       : f1_score(y_test, y_pred, zero_division=0),
            'precision': precision_score(y_test, y_pred, zero_division=0),
            'recall'   : recall_score(y_test, y_pred, zero_division=0),
        }
        mlflow.log_metrics(metricas)
        mlflow.sklearn.log_model(modelo, artifact_path=nombre)
        resultados[nombre] = {'modelo': modelo, 'y_prob': y_prob, 'y_pred': y_pred, **metricas}
        print(f'  ROC-AUC={metricas["roc_auc"]:.4f}  PR-AUC={metricas["pr_auc"]:.4f}  F1={metricas["f1"]:.4f}')

print('\nExperimentos registrados en MLflow.')

## 5. Evaluación y selección del mejor modelo

In [ ]:
df_comp = pd.DataFrame(
    {k: {m: v for m, v in v.items() if m not in ['modelo','y_prob','y_pred']}
     for k, v in resultados.items()}
).T.round(4)

display(df_comp.style.highlight_max(axis=0, color='#d4edda').format('{:.4f}'))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Barras comparativas
df_comp[['roc_auc','pr_auc','f1']].plot(kind='bar', ax=axes[0], width=0.7)
axes[0].set_title('Comparativa de métricas', fontweight='bold')
axes[0].set_xticklabels(df_comp.index, rotation=15, ha='right')
axes[0].set_ylim(0, 1)
axes[0].legend(loc='lower right')
axes[0].set_ylabel('Score')

# Curvas ROC
for nombre, res in resultados.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    axes[1].plot(fpr, tpr, label=f"{nombre} ({res['roc_auc']:.3f})")
axes[1].plot([0,1],[0,1],'k--', linewidth=0.8)
axes[1].set_xlabel('Tasa de falsos positivos')
axes[1].set_ylabel('Tasa de verdaderos positivos')
axes[1].set_title('Curvas ROC', fontweight='bold')
axes[1].legend(fontsize=8)

# Curvas Precision-Recall
for nombre, res in resultados.items():
    prec, rec, _ = precision_recall_curve(y_test, res['y_prob'])
    axes[2].plot(rec, prec, label=f"{nombre} (AP={res['pr_auc']:.3f})")
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precisión')
axes[2].set_title('Curvas Precision-Recall', fontweight='bold')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('comparativa_modelos.png', bbox_inches='tight')
plt.show()

In [ ]:
mejor_nombre = max(resultados, key=lambda k: resultados[k]['roc_auc'])
mejor_modelo  = resultados[mejor_nombre]['modelo']
print(f'Mejor modelo: {mejor_nombre}')
print(f'  ROC-AUC={resultados[mejor_nombre]["roc_auc"]:.4f}  '
      f'PR-AUC={resultados[mejor_nombre]["pr_auc"]:.4f}  '
      f'F1={resultados[mejor_nombre]["f1"]:.4f}')

if hasattr(mejor_modelo, 'feature_importances_'):
    fi = pd.Series(mejor_modelo.feature_importances_, index=FEATURES).sort_values()
    fig, ax = plt.subplots(figsize=(8, 4))
    fi.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Importancia de variables — {mejor_nombre}', fontweight='bold')
    ax.set_xlabel('Importancia')
    plt.tight_layout()
    plt.savefig('importancia_variables.png', bbox_inches='tight')
    plt.show()
    display(fi.sort_values(ascending=False).to_frame('importancia').style.format('{:.4f}'))

### Observaciones y conclusiones sobre los modelos

**Gradient Boosting** es el mejor modelo con **ROC-AUC = 0.91** y **PR-AUC = 0.74**.

**Hallazgos principales:**

1. **El historial domina las predicciones**: `freq_compra` (57%) y `veces_comprado` (30%) concentran ~87% de la importancia. El comportamiento pasado es el mejor predictor de compra futura.

2. **Segmentación comercial aporta valor incremental**: `seg5_enc` y `seg4_enc` aportan ~3%, confirmando que clientes del mismo segmento tienen patrones similares.

3. **La regresión logística sirve como baseline**: ROC-AUC = 0.80, notablemente inferior, lo que confirma que la relación entre features y compra es no-lineal.

4. **Gradient Boosting vs Random Forest**: ambos tienen AUC similar (~0.91), pero Gradient Boosting tiene mayor precisión (0.72 vs 0.53), lo que reduce las recomendaciones incorrectas — comportamiento preferible en producción.

5. **El desbalance de clases está manejado**: con ~21% de positivos y `class_weight='balanced'`, los modelos no colapsan prediciendo siempre la clase mayoritaria.

## 6. Generación de recomendaciones Top-3

Con el mejor modelo entrenado, se predicen las probabilidades de compra para cada par (cliente, producto) usando el último mes como contexto. Se excluyen los productos que el cliente ya compró ese mes y se entregan los 3 con mayor probabilidad.

In [ ]:
df_recomendar = grid_df[grid_df['Mes'] == ultimo_mes].copy()
df_recomendar['prob_compra'] = mejor_modelo.predict_proba(df_recomendar[FEATURES])[:, 1]

df_nuevos = df_recomendar[df_recomendar['compro'] == 0].copy()

recomendaciones = (
    df_nuevos.sort_values(['Cliente', 'prob_compra'], ascending=[True, False])
    .groupby('Cliente').head(3)
    .reset_index(drop=True)
)
recomendaciones['ranking'] = recomendaciones.groupby('Cliente').cumcount() + 1
recomendaciones = recomendaciones.merge(
    clientes[['Cliente', 'Oficina de ventas',
               'Denominación Gr#Clientes 4', 'Denominación Gr#Clientes 5']],
    on='Cliente', how='left'
)

print(f'Recomendaciones para {recomendaciones["Cliente"].nunique():,} clientes')
cols = ['Cliente','ranking','CodProducto','prob_compra','Oficina de ventas','Denominación Gr#Clientes 4']
display(recomendaciones[recomendaciones['Cliente'].isin(
    recomendaciones['Cliente'].unique()[:5])][cols].reset_index(drop=True))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

recomendaciones['CodProducto'].value_counts().plot(
    kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Productos más recomendados (Top-3)', fontweight='bold')
axes[0].set_xlabel('Producto')
axes[0].set_ylabel('Frecuencia')
axes[0].tick_params(axis='x', rotation=0)

recomendaciones.groupby('CodProducto')['prob_compra'].mean().sort_values(ascending=False).plot(
    kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Probabilidad promedio por producto', fontweight='bold')
axes[1].set_xlabel('Producto')
axes[1].set_ylabel('Probabilidad promedio')
axes[1].tick_params(axis='x', rotation=0)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

plt.tight_layout()
plt.savefig('distribucion_recomendaciones.png', bbox_inches='tight')
plt.show()

## 7. Guardar resultados

In [ ]:
recomendaciones.to_csv('recomendaciones_top3.csv', index=False)
df_recomendar[['Cliente','CodProducto','prob_compra','compro']].to_csv(
    'probabilidades_completas.csv', index=False)
df_comp.to_csv('comparativa_modelos.csv')

print('Archivos guardados:')
print('  recomendaciones_top3.csv      — Top-3 por cliente')
print('  probabilidades_completas.csv  — Probabilidades todos los pares')
print('  comparativa_modelos.csv       — Métricas de los 3 modelos')
print()
print('Para ver los experimentos en MLflow ejecuta en esta carpeta:')
print('  mlflow ui --backend-store-uri sqlite:///mlflow.db')